In [3]:
from pyspark.sql import SparkSession, functions as F, types as T
from datetime import datetime
import time

spark = (SparkSession.builder.master("local[2]").appName("rdds").config("spark.driver.memory", "2g").getOrCreate())

sc = spark.sparkContext

In [4]:
df_green = spark.read.parquet("data/pq/green/*/*") \
    .select("lpep_pickup_datetime", "PULocationID", "total_amount")

rdd = df_green.rdd
print(type(rdd))
print("Partitions:", rdd.getNumPartitions())
rdd.take(3)

26/11/25 13:24:49 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/pq/green/*/*.
java.io.FileNotFoundException: File data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:401)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:209)
	at org.apache.spark.sql.catalyst.analysis.ResolveData

<class 'pyspark.core.rdd.RDD'>
Partitions: 3


[Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 22, 13, 18, 32), PULocationID=244, total_amount=22.8),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 23, 17, 54, 10), PULocationID=236, total_amount=12.25),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 19, 10, 23, 37), PULocationID=166, total_amount=5.3)]

In [5]:
start = datetime(2020, 1, 1)

def keep_2020_onwards(row):
    return row.lpep_pickup_datetime is not None and row.lpep_pickup_datetime >= start

def to_key_value(row):
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    key = (hour, row.PULocationID)
    value = (row.total_amount or 0.0, 1)        # (amount, trip count)
    return (key, value)

def add_values(a, b):
    return (a[0] + b[0], a[1] + b[1])

rdd_revenue = (
    rdd
    .filter(keep_2020_onwards)     # narrow
    .map(to_key_value)             # narrow: (key, value) pairs
    .reduceByKey(add_values)       # wide: shuffle by key
)

In [6]:
print(rdd_revenue.toDebugString().decode())

(3) PythonRDD[22] at RDD at PythonRDD.scala:59 []
 |  MapPartitionsRDD[21] at mapPartitions at PythonRDD.scala:197 []
 |  ShuffledRDD[20] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(3) PairwiseRDD[19] at reduceByKey at /home/velpu/tmpbuild/ipykernel_1848/256869096.py:19 []
    |  PythonRDD[18] at reduceByKey at /home/velpu/tmpbuild/ipykernel_1848/256869096.py:19 []
    |  MapPartitionsRDD[16] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |  MapPartitionsRDD[15] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |  SQLExecutionRDD[14] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |  MapPartitionsRDD[13] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |  MapPartitionsRDD[12] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |  FileScanRDD[11] at javaToPython at NativeMethodAccessorImpl.java:0 []


In [7]:
start_t = time.time()
print(rdd_revenue.take(5))
print("Rows:", rdd_revenue.count())
print(f"RDD version: {time.time() - start_t:.1f} s")

[((datetime.datetime(2020, 1, 23, 17, 0), 236), (71.02, 5)), ((datetime.datetime(2020, 1, 19, 10, 0), 166), (969.2699999999993, 55)), ((datetime.datetime(2020, 1, 26, 16, 0), 7), (682.1099999999998, 50)), ((datetime.datetime(2020, 1, 27, 23, 0), 65), (119.80999999999997, 10)), ((datetime.datetime(2020, 1, 26, 17, 0), 95), (444.59000000000015, 28))]


Rows: 744766
RDD version: 26.9 s


In [8]:
result_schema = T.StructType([
    T.StructField("hour", T.TimestampType()),
    T.StructField("zone", T.IntegerType()),
    T.StructField("revenue", T.DoubleType()),
    T.StructField("count", T.IntegerType()),
])

df_result = spark.createDataFrame(
    rdd_revenue.map(lambda kv: (kv[0][0], kv[0][1], kv[1][0], kv[1][1])),
    schema=result_schema,
)
df_result.show(5)

+-------------------+----+------------------+-----+
|               hour|zone|           revenue|count|
+-------------------+----+------------------+-----+
|2020-01-23 17:00:00| 236|             71.02|    5|
|2020-01-19 10:00:00| 166| 969.2699999999993|   55|
|2020-01-26 16:00:00|   7| 682.1099999999998|   50|
|2020-01-27 23:00:00|  65|119.80999999999997|   10|
|2020-01-26 17:00:00|  95|444.59000000000015|   28|
+-------------------+----+------------------+-----+
only showing top 5 rows


Traceback (most recent call last):                                              
  File "/home/velpu/projects/data-engineering-project1/06-batch/.venv/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 264, in manager
    code = worker(sock, authenticated)
  File "/home/velpu/projects/data-engineering-project1/06-batch/.venv/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 95, in worker
    outfile.flush()
    ~~~~~~~~~~~~~^^
BrokenPipeError: [Errno 32] Broken pipe


In [9]:
start_t = time.time()
df_dataframe = (
    spark.read.parquet("data/pq/green/*/*")
    .filter(F.col("lpep_pickup_datetime") >= "2020-01-01")
    .groupBy(F.date_trunc("hour", "lpep_pickup_datetime").alias("hour"),
             F.col("PULocationID").alias("zone"))
    .agg(F.sum("total_amount").alias("revenue"), F.count("*").alias("count"))
)
print("Rows:", df_dataframe.count())
print(f"DataFrame version: {time.time() - start_t:.1f} s")

26/11/25 13:28:42 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/pq/green/*/*.
java.io.FileNotFoundException: File data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:401)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:209)
	at org.apache.spark.sql.catalyst.analysis.ResolveData

Rows: 744766
DataFrame version: 9.2 s


In [10]:
pairs = rdd.filter(keep_2020_onwards).map(lambda row: (row.PULocationID, row.total_amount or 0.0))

start_t = time.time()
r1 = pairs.reduceByKey(lambda a, b: a + b).collect()
print(f"reduceByKey: {time.time() - start_t:.1f} s")

start_t = time.time()
r2 = pairs.groupByKey().mapValues(sum).collect()
print(f"groupByKey:  {time.time() - start_t:.1f} s")

reduceByKey: 15.9 s


groupByKey:  41.4 s


In [11]:
def count_rows(partition_iterator):
    yield sum(1 for _ in partition_iterator)

rdd.mapPartitions(count_rows).collect()

[1317800, 597023, 389694]

In [12]:
import pandas as pd

def predict_in_batch(partition_iterator):
    rows = list(partition_iterator)                  # this partition's rows
    if not rows:
        return
    df = pd.DataFrame(rows, columns=["pickup", "zone", "amount", "distance"])
    # "Load the model" would happen HERE: once per partition, not once per row
    df["predicted_minutes"] = df["distance"] * 5    # the "model", applied to the whole batch
    for row in df.itertuples(index=False):
        yield (row.zone, row.distance, row.predicted_minutes)

rdd_for_model = (
    spark.read.parquet("data/pq/green/2020/01")
    .select("lpep_pickup_datetime", "PULocationID", "total_amount", "trip_distance")
    .rdd
)
rdd_for_model.mapPartitions(predict_in_batch).take(5)

[(244, 5.22, 26.099999999999998),
 (236, 0.87, 4.35),
 (166, 0.63, 3.15),
 (152, 2.71, 13.55),
 (51, 2.13, 10.649999999999999)]